In [11]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)  # override=True forces reload
key = os.getenv("ANTHROPIC_API_KEY")
print(f"Key starts with: {key[:20] if key else 'NONE'}")

Key starts with: sk-ant-api03-IEO-oRK


In [12]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [13]:
import anthropic
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say 'API connected' only."}]
)
print(response.content[0].text)

API connected


In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Reload data and models
df = pd.read_csv('../data/processed/creditcard_features.csv')
feature_cols = [c for c in df.columns if c not in ['Class', 'Time', 'Amount']]
X = df[feature_cols]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reload neural network
class FraudDetectionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = FraudDetectionNet(input_dim=len(feature_cols)).to(device)
model.load_state_dict(torch.load('../src/fraud_model.pth', map_location=device))
model.eval()

X_test_t = torch.FloatTensor(X_test.values).to(device)
with torch.no_grad():
    nn_proba = model(X_test_t).squeeze().cpu().numpy()

print(f"✅ Model loaded | Test set: {len(X_test):,} transactions")
print(f"High risk transactions (>80%): {(nn_proba > 0.8).sum()}")

✅ Model loaded | Test set: 56,962 transactions
High risk transactions (>80%): 108


In [15]:
def generate_risk_narrative(transaction: dict, fraud_prob: float) -> str:
    """Generate plain-English risk explanation for a flagged transaction"""
    
    prompt = f"""You are a fraud risk analyst at a bank.

A transaction has been flagged by our ML model with a {fraud_prob:.1%} fraud probability.

Transaction details:
- Amount: €{transaction.get('Amount_scaled', 0):.2f} (scaled)
- Hour of day: {int(transaction.get('hour_of_day', 0))}:00
- Night transaction: {'Yes' if transaction.get('is_night', 0) else 'No'}
- Amount z-score vs recent history: {transaction.get('amount_zscore', 0):.2f}
- Amount vs 10-transaction rolling avg: €{transaction.get('amount_rolling_10', 0):.2f}

Top PCA risk signals:
- V14: {transaction.get('V14', 0):.3f}
- V4:  {transaction.get('V4', 0):.3f}
- V11: {transaction.get('V11', 0):.3f}
- V12: {transaction.get('V12', 0):.3f}

Write a 3-4 sentence risk assessment a compliance officer would read.
Be specific about which signals are most concerning.
End with a recommended action on its own line: 
DECISION: APPROVE, REVIEW, or BLOCK"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=250,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


# Run on top 5 highest risk transactions
X_test_df = X_test.copy().reset_index(drop=True)
X_test_df['fraud_prob'] = nn_proba
top_risk = X_test_df.nlargest(5, 'fraud_prob')

print("=== AI-GENERATED RISK NARRATIVES ===\n")
narratives = []
for idx, (_, row) in enumerate(top_risk.iterrows()):
    print(f"Transaction #{idx+1} | Fraud Probability: {row['fraud_prob']:.1%}")
    print("-" * 55)
    narrative = generate_risk_narrative(row.to_dict(), row['fraud_prob'])
    print(narrative)
    narratives.append({
        'transaction': idx+1,
        'fraud_prob': row['fraud_prob'],
        'narrative': narrative
    })
    print()

# Save narratives
import json
with open('../reports/risk_narratives.json', 'w') as f:
    json.dump(narratives, f, indent=2)
print("✅ Risk narratives saved to reports/")

=== AI-GENERATED RISK NARRATIVES ===

Transaction #1 | Fraud Probability: 100.0%
-------------------------------------------------------
## Fraud Risk Assessment

This transaction has been flagged at maximum model confidence (100% fraud probability), driven primarily by extreme anomalies in the PCA-derived behavioral features V12 (-10.834) and V4 (9.506), both of which deviate severely from normal transaction patterns and are among the strongest fraud indicators in this model's feature space. V14 (-9.374) is also critically elevated — this component is consistently associated with compromised card behavior in payment fraud literature and carries significant diagnostic weight. Although the transaction amount of €1.10 appears superficially low-risk, this is consistent with a **probe or test transaction** pattern, where fraudsters verify card validity with a micro-amount before executing larger withdrawals, further supported by a rolling average discrepancy of €371.33 suggesting this amou

In [16]:
import sqlite3
conn = sqlite3.connect('../data/processed/fraud_warehouse.db')
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables in database:", tables)
conn.close()

Tables in database: []


In [17]:
import pandas as pd

df = pd.read_csv('../data/processed/creditcard_features.csv')

# Save a 10K sample — stratified to keep fraud cases
fraud = df[df.Class == 1]  # Keep ALL 492 fraud cases
legit = df[df.Class == 0].sample(9508, random_state=42)  # Sample legitimate
sample = pd.concat([fraud, legit]).sample(frac=1, random_state=42)

sample.to_csv('../data/processed/sample_features.csv', index=False)
print(f"Sample saved: {sample.shape}")
print(f"Fraud cases: {sample.Class.sum()}")

Sample saved: (10000, 39)
Fraud cases: 492
